In [79]:
import os
import sys
import pandas as pd

from pathlib import Path
from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_txt, read_rds, read_jsonl, read_excel_sheets
from n_gram_tracing import (
    common_ngrams,
    filter_ngrams_with_outside_occurrences_in_both_texts,
    tokenize_to_tokens,
    find_all_token_ngram_spans,
    find_independent_token_ngram_spans,
    ngram_occurrence_stats
)
from utils import apply_temp_doc_id, build_metadata_df

In [80]:
# --- set your args (strings) ---
base_loc = "/Volumes/BCross"

if not os.path.exists(base_loc):
    print("Using Offline Location")
    base_loc = "/Users/user/Documents/uni_work_offline"
else:
    print("Using Volume Location")

CORPUS = "Yelp"
DATA_TYPE = "test"

KNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/known_raw.jsonl"
UNKNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/unknown_raw.jsonl"
METADATA_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/metadata.rds"
MODEL_LOC = f"{base_loc}/models/gpt2"

# file_loc = '/Users/user/Downloads/-as_oxYIDXEGRxLxgd6DGA vs -AXo3Eeg-Ipqx54f8ZahLQ.xlsx'
# PROBLEM = Path(file_loc).stem
PROBLEM = "9q36VyX34T8C9hf2K88afw vs 9qyLpg3JTVUZXS4XrQgZkQ"

print(f"Problem: {PROBLEM}")

Using Volume Location
Problem: 9q36VyX34T8C9hf2K88afw vs 9qyLpg3JTVUZXS4XrQgZkQ


In [81]:
tokenizer, model = load_model(MODEL_LOC)

In [82]:
known = read_jsonl(KNOWN_LOC)
known = apply_temp_doc_id(known)
    
unknown = read_jsonl(UNKNOWN_LOC)
unknown = apply_temp_doc_id(unknown)

metadata = read_rds(METADATA_LOC)
filtered_metadata = metadata[
    (metadata['corpus'] == CORPUS)
    & (metadata['problem'] ==  PROBLEM)
]

agg_metadata = build_metadata_df(filtered_metadata, known, unknown)

# -----
# Get the chosen text & metadata
# -----
    
known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]

selected_known = known[known['author'] == known_author]
selected_unknown = unknown[unknown['author'] == unknown_author]

unknown_doc = selected_unknown['doc_id'].iloc[0]
unknown_text = selected_unknown['text'].iloc[0]

num_known_problems = len(selected_known)
known_texts = selected_known['text'].tolist()
print(f"There are {num_known_problems} known texts in the problem")

There are 5 known texts in the problem


In [83]:
def dedupe_ngrams(ngrams):
    """
    Deduplicate n-grams while preserving order.
    """
    return [list(x) for x in dict.fromkeys(tuple(g) for g in ngrams)]


def sort_ngrams(ngrams):
    """
    Sort first by number of tokens, then by total character length.
    """
    return sorted(
        ngrams,
        key=lambda x: (len(x), sum(len(str(token)) for token in x))
    )

In [84]:
def current_pair_local_greatest_common(
    known_texts,
    unknown_text,
    tokenizer,
    *,
    lowercase=True,
):
    kept_ngrams = []

    for known_text in known_texts:
        common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=lowercase,
        )

        pair_kept = filter_ngrams_with_outside_occurrences_in_both_texts(
            ngrams=common,
            known_text=known_text,
            unknown_text=unknown_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        )

        kept_ngrams.extend(pair_kept)

    kept_ngrams = dedupe_ngrams(kept_ngrams)
    kept_ngrams = sort_ngrams(kept_ngrams)
    
    return kept_ngrams

In [85]:
def global_second_pass_greatest_common(
    known_texts,
    unknown_text,
    tokenizer,
    *,
    lowercase=True,
):
    all_common = []

    # First pass: collect all pairwise common n-grams
    for known_text in known_texts:
        pair_common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=lowercase,
        )

        all_common.extend(pair_common)

    # Problem-level candidate list
    global_common = dedupe_ngrams(all_common)
    global_common = sort_ngrams(global_common)

    if not global_common:
        return []

    unknown_stats = ngram_occurrence_stats(
        ngrams=global_common,
        text=unknown_text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    known_stats_list = [
        ngram_occurrence_stats(
            ngrams=global_common,
            text=known_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        )
        for known_text in known_texts
    ]

    kept = []

    for g in dict.fromkeys(tuple(x) for x in global_common):
        unknown_keep = unknown_stats.get(g, {}).get("keep", False)

        known_keep = any(
            stats.get(g, {}).get("keep", False)
            for stats in known_stats_list
        )

        if unknown_keep and known_keep:
            kept.append(list(g))

    kept = sort_ngrams(kept)
    
    return kept

In [46]:
def effective_unknown_scored_ngrams_after_greatest_common(
    ngrams,
    unknown_text,
    tokenizer,
    *,
    lowercase=True,
):
    """
    Given a final n-gram list, return the n-grams that would still have
    at least one uncovered occurrence in the unknown document.

    This approximates the extra effect of using greatest_common=True
    inside score_ngrams_to_df.
    """
    stats = ngram_occurrence_stats(
        ngrams=ngrams,
        text=unknown_text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    kept = [
        row["ngram"]
        for row in stats.values()
        if row["outside_count"] > 0
    ]

    return sort_ngrams(kept)

In [47]:
current_kept_ngrams = current_pair_local_greatest_common(
    known_texts=known_texts,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

global_kept_ngrams = global_second_pass_greatest_common(
    known_texts=known_texts,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

print(f"Current pair-local version kept: {len(current_kept_ngrams)} n-grams")
print(f"Global second-pass version kept: {len(global_kept_ngrams)} n-grams")

current_set = set(tuple(x) for x in current_kept_ngrams)
global_set = set(tuple(x) for x in global_kept_ngrams)

print(f"In current only: {len(current_set - global_set)}")
print(f"In global only: {len(global_set - current_set)}")
print(f"In both: {len(current_set & global_set)}")

Current pair-local version kept: 53 n-grams
Global second-pass version kept: 48 n-grams
In current only: 5
In global only: 0
In both: 48


In [48]:
current_set - global_set

{('Ġall', 'Ġthe'), ('Ġat',), ('Ġhave',), ('Ġit',), ('Ġthere',)}

In [49]:

current_effective_scored_ngrams = effective_unknown_scored_ngrams_after_greatest_common(
    ngrams=current_kept_ngrams,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

global_effective_scored_ngrams = effective_unknown_scored_ngrams_after_greatest_common(
    ngrams=global_kept_ngrams,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

print(f"Current candidate n-grams before scoring cleanup: {len(current_kept_ngrams)}")
print(f"Current effective n-grams after scoring cleanup: {len(current_effective_scored_ngrams)}")
print(f"Global candidate n-grams before scoring cleanup: {len(global_kept_ngrams)}")
print(f"Global effective n-grams after scoring cleanup: {len(global_effective_scored_ngrams)}")

Current candidate n-grams before scoring cleanup: 53
Current effective n-grams after scoring cleanup: 48
Global candidate n-grams before scoring cleanup: 48
Global effective n-grams after scoring cleanup: 48


In [50]:
def compare_ngram_lists(before, after, label=""):
    before_set = set(tuple(x) for x in before)
    after_set = set(tuple(x) for x in after)

    removed = before_set - after_set
    kept = before_set & after_set

    print(f"\n{label}")
    print(f"Before: {len(before_set)}")
    print(f"After: {len(after_set)}")
    print(f"Kept: {len(kept)}")
    print(f"Removed by scoring-stage greatest_common: {len(removed)}")

    return {
        "kept": sort_ngrams([list(x) for x in kept]),
        "removed": sort_ngrams([list(x) for x in removed]),
    }

In [51]:
current_comparison = compare_ngram_lists(
    current_kept_ngrams,
    current_effective_scored_ngrams,
    label="Current pair-local method",
)

global_comparison = compare_ngram_lists(
    global_kept_ngrams,
    global_effective_scored_ngrams,
    label="Global second-pass method",
)


Current pair-local method
Before: 53
After: 48
Kept: 48
Removed by scoring-stage greatest_common: 5

Global second-pass method
Before: 48
After: 48
Kept: 48
Removed by scoring-stage greatest_common: 0


In [52]:
current_removed = current_comparison["removed"]
global_removed = global_comparison["removed"]

print(current_removed[:20])
print(global_removed[:20])

[['Ġit'], ['Ġat'], ['Ġhave'], ['Ġthere'], ['Ġall', 'Ġthe']]
[]


In [53]:
import html
from typing import Any, List, Sequence, Optional, Tuple

from IPython.display import display, HTML

from n_gram_tracing import (
    tokenize_to_tokens,
    tokens_to_text,
    find_all_token_ngram_spans,
    find_independent_token_ngram_spans,
)


def _normalise_ngram_list(ngrams):
    """
    Accept either:
      - a single n-gram like ['Ġwe']
      - a list of n-grams like [['Ġwe'], ['Ġwe', 'Ġhave']]
    and always return a list of n-grams.
    """
    if not ngrams:
        return []

    # single ngram passed directly
    if isinstance(ngrams[0], str):
        return [list(ngrams)]

    return [list(g) for g in ngrams]


def _collect_ngram_spans(
    text: str,
    ngrams: Sequence[Sequence[Any]],
    *,
    tokenizer: Any,
    lowercase: bool = True,
    greatest_common: bool = False,
    all_ngrams: Optional[Sequence[Sequence[Any]]] = None,
) -> Tuple[List[Any], List[dict]]:
    """
    Return tokenized text plus span records for all requested n-grams.
    """
    tokens = tokenize_to_tokens(text, tokenizer=tokenizer, lowercase=lowercase)

    span_rows = []

    for ng_idx, ng in enumerate(ngrams):
        if greatest_common:
            if all_ngrams is None:
                raise ValueError("all_ngrams must be provided when greatest_common=True")

            spans = find_independent_token_ngram_spans(
                tokens=tokens,
                ngram_tokens=list(ng),
                all_ngrams=all_ngrams,
                allow_overlaps=True,
            )
        else:
            spans = find_all_token_ngram_spans(
                tokens=tokens,
                ngram_tokens=list(ng),
                allow_overlaps=True,
            )

        for s, e in spans:
            span_rows.append({
                "start": s,
                "end": e,
                "ngram": list(ng),
                "ngram_idx": ng_idx,
                "length": e - s,
            })

    return tokens, span_rows


def _render_highlighted_html(
    tokens: List[Any],
    selected_spans: List[dict],
    tokenizer: Any,
    *,
    title: str = "",
    palette: Optional[List[str]] = None,
) -> str:
    """
    Render tokens with highlighted spans as HTML.

    Keeps the original compact formatting, but:
      - uses brighter colours for dark mode
      - allows overlapping highlights by splitting the token background
    """
    if palette is None:
        palette = [
            "#facc15",  # amber
            "#22c55e",  # green
            "#38bdf8",  # sky blue
            "#fb7185",  # rose
            "#a78bfa",  # violet
            "#f97316",  # orange
            "#2dd4bf",  # teal
            "#e879f9",  # fuchsia
        ]

    def colour_for_idx(i):
        return palette[i % len(palette)]

    def stacked_background(colours):
        colours = list(dict.fromkeys(colours))

        if len(colours) == 1:
            return colours[0]

        n = len(colours)
        parts = []

        for i, colour in enumerate(colours):
            start = 100 * i / n
            end = 100 * (i + 1) / n
            parts.append(f"{colour} {start:.1f}% {end:.1f}%")

        return "linear-gradient(to bottom, " + ", ".join(parts) + ")"

    html_parts = []

    if title:
        html_parts.append(
            f"<h4 style='margin-bottom:8px; color:#e5e7eb;'>{html.escape(str(title))}</h4>"
        )

    # legend
    if selected_spans:
        seen = []

        for row in selected_spans:
            if row["ngram"] not in seen:
                seen.append(row["ngram"])

        legend_bits = []

        for ng in seen:
            # use the ngram_idx from the first matching span so legend colour matches body
            first_idx = next(
                row["ngram_idx"]
                for row in selected_spans
                if row["ngram"] == ng
            )
            colour = colour_for_idx(first_idx)
            label = tokens_to_text(ng, tokenizer)

            legend_bits.append(
                f"<span style='"
                f"background:{colour}; "
                f"color:#111827; "
                f"padding:2px 6px; "
                f"border-radius:4px; "
                f"margin-right:8px; "
                f"font-weight:700;"
                f"'>"
                f"{html.escape(label)}"
                f"</span>"
            )

        html_parts.append(
            "<div style='margin-bottom:10px;'>" + "".join(legend_bits) + "</div>"
        )

    # map each token index to all n-grams covering it
    token_hits = [[] for _ in tokens]

    for row in selected_spans:
        for idx in range(row["start"], row["end"]):
            if 0 <= idx < len(tokens):
                token_hits[idx].append(row)

    # body rendered token-by-token, so overlaps can be shown
    body_parts = []

    for tok_idx, tok in enumerate(tokens):
        tok_text = html.escape(tokens_to_text([tok], tokenizer))
        hits = token_hits[tok_idx]

        if not hits:
            body_parts.append(tok_text)
            continue

        colours = [
            colour_for_idx(hit["ngram_idx"])
            for hit in hits
        ]

        hover_text = " | ".join(
            tokens_to_text(hit["ngram"], tokenizer)
            for hit in hits
        )

        bg = stacked_background(colours)

        body_parts.append(
            f"<span title='{html.escape(hover_text)}' style='"
            f"background:{bg}; "
            f"color:#111827; "
            f"padding:1px 2px; "
            f"border-radius:3px; "
            f"font-weight:700; "
            f"box-shadow: inset 0 0 0 1px rgba(0,0,0,0.25);"
            f"'>"
            f"{tok_text}"
            f"</span>"
        )

    html_parts.append(
        "<div style='white-space:pre-wrap; line-height:1.6; "
        "border:1px solid #374151; "
        "background:#111827; "
        "color:#e5e7eb; "
        "padding:12px; "
        "border-radius:8px; "
        "font-family:Arial, sans-serif; "
        "font-size:14px;'>"
        + "".join(body_parts) +
        "</div>"
    )

    return "".join(html_parts)


def show_ngram_highlights(
    *,
    unknown_text: str,
    known_texts: Sequence[str],
    tokenizer: Any,
    ngrams,
    known_doc_ids: Optional[Sequence[Any]] = None,
    unknown_doc_id: Optional[Any] = None,
    lowercase: bool = True,
    greatest_common: bool = False,
    all_ngrams: Optional[Sequence[Sequence[Any]]] = None,
):
    """
    Display highlighted occurrences for:
      1. unknown text
      2. each known text

    Parameters
    ----------
    ngrams:
        either a single n-gram like ['Ġwe']
        or a list like [['Ġwe'], ['Ġwe', 'Ġhave']]

    greatest_common:
        if True, use find_independent_token_ngram_spans()

    all_ngrams:
        required when greatest_common=True
    """
    ngrams = _normalise_ngram_list(ngrams)

    if known_doc_ids is None:
        known_doc_ids = [f"known_{i+1}" for i in range(len(known_texts))]

    # unknown first
    unk_tokens, unk_spans = _collect_ngram_spans(
        unknown_text,
        ngrams,
        tokenizer=tokenizer,
        lowercase=lowercase,
        greatest_common=greatest_common,
        all_ngrams=all_ngrams,
    )

    unk_title = "Unknown"
    if unknown_doc_id is not None:
        unk_title += f" | doc_id={unknown_doc_id}"

    display(HTML(_render_highlighted_html(
        unk_tokens,
        unk_spans,
        tokenizer,
        title=unk_title,
    )))

    # then all known docs
    for text, doc_id in zip(known_texts, known_doc_ids):
        tokens, spans = _collect_ngram_spans(
            text,
            ngrams,
            tokenizer=tokenizer,
            lowercase=lowercase,
            greatest_common=greatest_common,
            all_ngrams=all_ngrams,
        )

        title = f"Known | doc_id={doc_id}"

        display(HTML(_render_highlighted_html(
            tokens,
            spans,
            tokenizer,
            title=title,
        )))

In [54]:
def find_parent_ngrams(target_ngram, candidate_ngrams):
    """
    Find longer n-grams in candidate_ngrams that contain target_ngram.
    """
    target = tuple(target_ngram)
    parents = []

    for g in candidate_ngrams:
        g = tuple(g)

        if len(g) <= len(target):
            continue

        for i in range(len(g) - len(target) + 1):
            if g[i:i + len(target)] == target:
                parents.append(list(g))
                break

    return parents

def find_parent_ngrams_with_offsets(target_ngram, candidate_ngrams):
    """
    Find longer n-grams in candidate_ngrams that contain target_ngram.

    Returns:
        [
            {
                "parent_ngram": [...],
                "offset": int
            },
            ...
        ]
    """
    target = tuple(target_ngram)
    parents = []

    for g in candidate_ngrams:
        parent = tuple(g)

        if len(parent) <= len(target):
            continue

        for offset in range(len(parent) - len(target) + 1):
            if parent[offset:offset + len(target)] == target:
                parents.append({
                    "parent_ngram": list(parent),
                    "offset": offset,
                })

    return parents


def print_ngrams_with_parents(result_tokens, *, max_parents=None):
    """
    Print only n-grams from result_tokens that are subgrams of longer n-grams
    also in result_tokens.

    Output includes:
      - the original/child n-gram
      - its parent n-grams
      - a copy-paste block for show_ngram_highlights()
    """
    grams = [
        list(g)
        for g in dict.fromkeys(tuple(g) for g in result_tokens)
    ]

    found_any = False

    for child in grams:
        parents = find_parent_ngrams_with_offsets(child, grams)

        if not parents:
            continue

        found_any = True

        print("\n" + "=" * 80)
        print("CHILD:")
        print(child)

        print("\nPARENTS:")
        shown = parents if max_parents is None else parents[:max_parents]

        for p in shown:
            print(f"offset {p['offset']}: {p['parent_ngram']}")

        if max_parents is not None and len(parents) > max_parents:
            print(f"... plus {len(parents) - max_parents} more")

        print("\nCOPY INTO HIGHLIGHTER:")
        print("[")
        print(f"    {child},")
        for p in shown:
            print(f"    {p['parent_ngram']},")
        print("]")

    if not found_any:
        print("No n-grams have parent n-grams in this list.")
        
def show_ngram_occurrences(
    text,
    target_ngram,
    candidate_ngrams,
    tokenizer,
    *,
    lowercase=True,
    window=12,
    label="text",
):
    """
    Print occurrences of target_ngram, showing whether each occurrence is
    inside a longer candidate n-gram.
    """
    tokens = tokenize_to_tokens(
        text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    target_spans = find_all_token_ngram_spans(
        tokens=tokens,
        ngram_tokens=target_ngram,
        allow_overlaps=True,
    )

    parent_ngrams = find_parent_ngrams(target_ngram, candidate_ngrams)

    parent_spans = []

    for parent in parent_ngrams:
        spans = find_all_token_ngram_spans(
            tokens=tokens,
            ngram_tokens=parent,
            allow_overlaps=True,
        )

        for span in spans:
            parent_spans.append({
                "parent_ngram": parent,
                "span": span,
            })

    print("\n" + "=" * 80)
    print(label)
    print(f"Target n-gram: {target_ngram}")
    print(f"Target occurrences: {len(target_spans)}")
    print(f"Number of possible parent n-grams: {len(parent_ngrams)}")

    if parent_ngrams:
        print("\nParent n-grams containing target:")
        for p in parent_ngrams[:20]:
            print(" ", p)

        if len(parent_ngrams) > 20:
            print(f"  ... plus {len(parent_ngrams) - 20} more")

    print("\nOccurrences:")

    for occ_i, (s, e) in enumerate(target_spans, start=1):
        left = max(0, s - window)
        right = min(len(tokens), e + window)

        context_tokens = tokens[left:right]
        context_text = tokens_to_text(context_tokens, tokenizer)

        blockers = [
            p
            for p in parent_spans
            if p["span"][0] <= s and e <= p["span"][1]
        ]

        print("\n---")
        print(f"Occurrence {occ_i}: span=({s}, {e})")
        print(f"Blocked by longer n-gram? {len(blockers) > 0}")
        print(f"Context: {context_text}")

        if blockers:
            print("Blocking parent spans:")
            for b in blockers[:10]:
                print(f"  span={b['span']} parent={b['parent_ngram']}")

            if len(blockers) > 10:
                print(f"  ... plus {len(blockers) - 10} more")

In [55]:
# import ast

# data = read_excel_sheets(file_loc, sheet_names=["phrase score"])
# data = data['phrase score']

# result_tokens = data["tokens"].apply(ast.literal_eval).tolist()

# print(f"Number of Tokens in Result: {len(result_tokens)}")

In [69]:
result_tokens = current_comparison['kept']
# print_ngrams_with_parents(result_tokens)

In [65]:
current_removed

[['Ġit'], ['Ġat'], ['Ġhave'], ['Ġthere'], ['Ġall', 'Ġthe']]

In [75]:
n_gram_for_parent = ['Ġall', 'Ġthe']
parent_list = find_parent_ngrams(n_gram_for_parent, result_tokens)

full_parent_list = [n_gram_for_parent] + parent_list
print(full_parent_list)

n_gram_to_check = (
full_parent_list
)

show_ngram_highlights(
    unknown_text=unknown_text,
    known_texts=selected_known["text"].tolist(),
    known_doc_ids=selected_known["doc_id"].tolist(),
    unknown_doc_id=unknown_doc,
    tokenizer=tokenizer,
    ngrams=n_gram_to_check,
    lowercase=True,
    greatest_common=True,
    all_ngrams=result_tokens,
)

[['Ġall', 'Ġthe'], ['Ġof', 'Ġall', 'Ġthe']]
